# NBME - DeBERTa-v3-base Training (Colab)

基於 [yasufuminakama](https://www.kaggle.com/yasufuminakama/nbme-deberta-v3-large-baseline-train) 的 notebook 修改：
- 模型：`microsoft/deberta-v3-base`
- 資料來源：讀 `train_preprocessed_5fold.pkl`（已含 fold 切分）
- 環境：Google Colab Pro

## 整體流程
```
讀 pkl → feature_text 前處理 → tokenize → 建立 token-level label
→ 5-fold 訓練 → 每 epoch 計算 char-level F1 → 存最佳模型
→ 輸出 oof_df.pkl（供 inference notebook 調 threshold 用）
```

# Colab Setup

In [148]:
# transformers：模型與 tokenizer 主要套件
# sentencepiece：DeBERTa-v3 tokenizer 的必要相依套件
!pip install -q transformers sentencepiece

from google.colab import drive
drive.mount('/content/drive')

#---
import pickle
import pandas as pd

BASE = '/content/drive/MyDrive/NBME_Score-Clinical-Patient-Notes/data/nbme-score-clinical-patient-notes/processed'

# 暫時修補 StringDtype，讓舊版 pkl 可以被新版 pandas 讀取
_orig_init = pd.StringDtype.__init__
pd.StringDtype.__init__ = lambda self, *args, **kwargs: _orig_init(self)

with open(f'{BASE}/train_preprocessed_5fold.pkl', 'rb') as f:
    train = pickle.load(f)
with open(f'{BASE}/test_preprocessed.pkl', 'rb') as f:
    test = pickle.load(f)

# 還原
pd.StringDtype.__init__ = _orig_init

# 把 StringDtype 欄位轉成 object，避免之後再出問題
for col in train.select_dtypes(include='string').columns:
    train[col] = train[col].astype(str)
for col in test.select_dtypes(include='string').columns:
    test[col] = test[col].astype(str)

# 覆蓋存回 Drive
train.to_pickle(f'{BASE}/train_preprocessed_5fold.pkl')
test.to_pickle(f'{BASE}/test_preprocessed.pkl')

print('Done!')
print(train.dtypes)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Done!
id                 object
case_num            int64
pn_num             object
feature_num        object
pn_history         object
feature_text       object
annotation_text    object
char_spans         object
fold                int64
dtype: object


# Directory Settings

In [149]:
import os

# ========== 請修改成你的路徑 ==========
BASE_DIR   = '/content/drive/MyDrive/NBME_Score-Clinical-Patient-Notes'
PKL_PATH   = BASE_DIR + '/data/nbme-score-clinical-patient-notes/processed/train_preprocessed_5fold.pkl'
OUTPUT_DIR = BASE_DIR + '/outputs/deberta-v3-base/'   # 模型權重、log、oof_df 都存在這裡
# =====================================

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'OUTPUT_DIR: {OUTPUT_DIR}')

OUTPUT_DIR: /content/drive/MyDrive/NBME_Score-Clinical-Patient-Notes/outputs/deberta-v3-base/


# CFG

所有超參數集中在這裡，方便調整。

In [ ]:
class CFG:
    debug=False

    apex=False   # DeBERTa-v3 在 PyTorch 2.6 下 fp16/bf16 均有問題，用 float32
    num_workers = 2

    model    = 'microsoft/deberta-v3-large'   # base 模型 hidden state 訊噪比太低，改用 large

    scheduler        = 'cosine'
    batch_scheduler  = True
    num_cycles       = 0.5
    num_warmup_steps = 0

    epochs     = 5
    encoder_lr = 2e-5
    decoder_lr = 2e-5
    min_lr     = 1e-6
    eps        = 1e-6
    betas      = (0.9, 0.999)
    batch_size = 8      # large 模型較大，batch size 從 16 降到 8
    fc_dropout = 0.2
    max_len    = 512
    weight_decay = 0.01
    gradient_accumulation_steps = 2   # 等效 batch size = 8 * 2 = 16
    max_grad_norm = 1000

    seed       = 42
    n_fold     = 5
    trn_fold   = [0, 1, 2, 3, 4]
    train      = True
    print_freq = 100

if CFG.debug:
    CFG.epochs   = 1
    CFG.trn_fold = [0]

# Library

In [151]:
import os
import gc
import re
import ast
import time
import math
import random
import itertools
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

from tqdm.auto import tqdm
from sklearn.metrics import f1_score

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset

import tokenizers
import transformers
print(f'tokenizers.__version__: {tokenizers.__version__}')
print(f'transformers.__version__: {transformers.__version__}')

from transformers import AutoModel, AutoConfig
from transformers import DebertaV2TokenizerFast   # deberta-v3 專用 fast tokenizer
from transformers import get_linear_schedule_with_warmup, get_cosine_schedule_with_warmup

%env TOKENIZERS_PARALLELISM=true

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {device}')

tokenizers.__version__: 0.22.2
transformers.__version__: 5.0.0
env: TOKENIZERS_PARALLELISM=true
device: cuda


# Scoring Functions

競賽使用 **character-level micro F1** 作為評估指標：
- 把預測的 span 和真實的 span 都展開成字元層級的 0/1 陣列
- 再計算 micro F1（跨所有樣本合併計算）

例：
```
真實 span: [(203, 217)]  → binary: 000...0111...1000...0
預測 span: [(203, 215)]  → binary: 000...0111...1000...0（少預測 2 個字元）
→ TP=12, FP=0, FN=2 → F1 = 2*12 / (2*12+0+2) ≈ 0.923
```

In [152]:
def micro_f1(preds, truths):
    """把所有樣本的預測和真實拼起來，計算整體 binary F1"""
    preds  = np.concatenate(preds)
    truths = np.concatenate(truths)
    return f1_score(truths, preds)


def spans_to_binary(spans, length=None):
    """把 span list（例如 [[203,217],[300,315]]）轉成字元層級的 0/1 陣列"""
    length = np.max(spans) if length is None else length
    binary = np.zeros(length)
    for start, end in spans:
        binary[start:end] = 1
    return binary


def span_micro_f1(preds, truths):
    """計算 span-level 的 micro F1（競賽官方指標）"""
    bin_preds  = []
    bin_truths = []
    for pred, truth in zip(preds, truths):
        # 預測和真實都是空的，這筆不影響分數，直接跳過
        if not len(pred) and not len(truth):
            continue
        length = max(
            np.max(pred)  if len(pred)  else 0,
            np.max(truth) if len(truth) else 0
        )
        bin_preds.append(spans_to_binary(pred, length))
        bin_truths.append(spans_to_binary(truth, length))
    return micro_f1(bin_preds, bin_truths)


def create_labels_for_scoring(df):
    """
    從 pkl 的 char_spans 欄位建立 ground truth
    輸入：char_spans = [(70, 91), (176, 183)]
    輸出：[[70, 91], [176, 183]]
    """
    truths = []
    for char_spans in df['char_spans'].values:
        truth = [[start, end] for start, end in char_spans]
        truths.append(truth)
    return truths


def get_char_probs(texts, predictions, tokenizer):
    """
    把模型輸出的 token-level 機率對應回 character-level 機率

    每個 token 有一個預測機率，對應到原始文字的某個字元範圍（offset_mapping）
    這個函數把該範圍內的所有字元都填上這個機率值
    """
    results = [np.zeros(len(t)) for t in texts]
    for i, (text, prediction) in enumerate(zip(texts, predictions)):
        encoded = tokenizer(text, add_special_tokens=True, return_offsets_mapping=True)
        for offset_mapping, pred in zip(encoded['offset_mapping'], prediction):
            start, end = offset_mapping
            results[i][start:end] = pred
    return results


def get_results(char_probs, th=0.3):
    """
    character 機率 → span 字串

    步驟：
    1. 找出機率 >= threshold 的字元位置
    2. 把連續的位置合併成一個 span
    3. 多個 span 用分號串接，例如 '70 91;176 183'
    """
    results = []
    for char_prob in char_probs:
        result = np.where(char_prob >= th)[0] + 1
        result = [list(g) for _, g in itertools.groupby(
            result, key=lambda n, c=itertools.count(): n - next(c))]
        result = [f'{min(r)} {max(r)}' for r in result]
        results.append(';'.join(result))
    return results


def get_predictions(results):
    """span 字串 → list of [[start, end], ...]"""
    predictions = []
    for result in results:
        prediction = []
        if result != '':
            for loc in [s.split() for s in result.split(';')]:
                start, end = int(loc[0]), int(loc[1])
                prediction.append([start, end])
        predictions.append(prediction)
    return predictions

# Utils

In [153]:
def get_score(y_true, y_pred):
    return span_micro_f1(y_true, y_pred)


def get_logger(filename=OUTPUT_DIR + 'train'):
    """同時輸出到 console 和 .log 檔，方便之後回頭看訓練紀錄"""
    from logging import getLogger, INFO, StreamHandler, FileHandler, Formatter
    logger = getLogger(__name__)
    logger.setLevel(INFO)
    handler1 = StreamHandler()
    handler1.setFormatter(Formatter('%(message)s'))
    handler2 = FileHandler(filename=f'{filename}.log')
    handler2.setFormatter(Formatter('%(message)s'))
    logger.addHandler(handler1)
    logger.addHandler(handler2)
    return logger

LOGGER = get_logger()


def seed_everything(seed=42):
    """固定所有隨機種子，確保訓練結果可以重現"""
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(seed=CFG.seed)

# Data Loading

In [154]:
# 直接讀前處理好的 pkl（已合併三表 + 解析 char_spans + 5-fold 切分）
train = pd.read_pickle(PKL_PATH)

print(f'train.shape: {train.shape}')
print(f'columns: {train.columns.tolist()}')
display(train.head(3))
print('fold distribution:')
print(train['fold'].value_counts().sort_index())

train.shape: (14300, 9)
columns: ['id', 'case_num', 'pn_num', 'feature_num', 'pn_history', 'feature_text', 'annotation_text', 'char_spans', 'fold']


,id,case_num,pn_num,feature_num,pn_history,feature_text,annotation_text,char_spans,fold
0,00016_000,0,00016,000,HPI: 17yo M presents with palpitations. Patien...,Family-history-of-MI-OR-Family-history-of-myoc...,[dad with recent heart attcak],"[(696, 724)]",0
1,00016_001,0,00016,001,HPI: 17yo M presents with palpitations. Patien...,Family-history-of-thyroid-disorder,"[mom with ""thyroid disease]","[(668, 693)]",0
2,00016_002,0,00016,002,HPI: 17yo M presents with palpitations. Patien...,Chest-pressure,[chest pressure],"[(203, 217)]",0


fold distribution:
fold
0    2839
1    2868
2    2860
3    2880
4    2853
Name: count, dtype: int64


In [155]:
def process_feature_text(text):
    """
    feature_text 前處理：把 dash 換成空白，讓模型更容易理解語意

    原始：'Family-history-of-MI-OR-Family-history-of-myocardial-infarction'
    處理後：'Family history of MI or Family history of myocardial infarction'
    """
    text = re.sub('I-year', '1-year', text)  # 修正 OCR 錯誤：I -> 1
    text = re.sub('-OR-', ' or ', text)       # -OR- 換成自然語言的 or
    text = re.sub('-', ' ', text)             # 其餘的 dash 換成空白
    return text

train['feature_text'] = train['feature_text'].apply(process_feature_text)

# annotation_length：這筆資料有幾段答案 span（0 表示無答案，約 31% 的資料）
train['annotation_length'] = train['char_spans'].apply(len)

print(train['annotation_length'].value_counts().sort_index())
display(train[['feature_text', 'char_spans', 'annotation_length']].head(5))

annotation_length
0     4399
1     7153
2     1856
3      484
4      224
5       72
6       48
7       22
8       17
9        8
10       5
11       4
12       5
14       1
16       1
17       1
Name: count, dtype: int64


,feature_text,char_spans,annotation_length
0,Family history of MI or Family history of myoc...,"[(696, 724)]",1
1,Family history of thyroid disorder,"[(668, 693)]",1
2,Chest pressure,"[(203, 217)]",1
3,Intermittent symptoms,"[(70, 91), (176, 183)]",2
4,Lightheaded,"[(222, 258)]",1


In [156]:
# debug 模式：只取 500 筆，快速確認整個 pipeline 不會報錯
if CFG.debug:
    train = train.sample(n=500, random_state=0).reset_index(drop=True)
    print(f'Debug mode: {len(train)} samples')
    print(train['fold'].value_counts().sort_index())

# Tokenizer

DeBERTa-v3 系列必須使用 `DebertaV2TokenizerFast`，不能用一般的 `AutoTokenizer`。

tokenizer 會把文字轉成 token id，並記錄每個 token 對應到原始文字的哪個字元範圍（`offset_mapping`），
這個資訊在建立 label 和把預測結果對應回字元時都會用到。

In [157]:
tokenizer = DebertaV2TokenizerFast.from_pretrained(CFG.model)
tokenizer.save_pretrained(OUTPUT_DIR + 'tokenizer/')  # 存起來供 inference notebook 使用
CFG.tokenizer = tokenizer
print('Tokenizer loaded.')

Tokenizer loaded.


# Define max_len

動態計算最適合的 `max_len`：
- 找出所有 `pn_history` 中 token 數最長的那筆
- 加上所有 `feature_text` 中 token 數最長的那筆
- 再加 3 個特殊 token（`[CLS]`, `[SEP]`, `[SEP]`）

這樣可以確保所有資料都不被截斷，同時不浪費記憶體。

In [158]:
pn_history_lengths = []
for text in tqdm(train['pn_history'].fillna('').unique(), desc='pn_history lengths'):
    length = len(tokenizer(text, add_special_tokens=False)['input_ids'])
    pn_history_lengths.append(length)
LOGGER.info(f'pn_history max token length: {max(pn_history_lengths)}')

feature_lengths = []
for text in tqdm(train['feature_text'].unique(), desc='feature_text lengths'):
    length = len(tokenizer(text, add_special_tokens=False)['input_ids'])
    feature_lengths.append(length)
LOGGER.info(f'feature_text max token length: {max(feature_lengths)}')

# [CLS] pn_history_tokens [SEP] feature_tokens [SEP] → +3
CFG.max_len = max(pn_history_lengths) + max(feature_lengths) + 3
CFG.max_len = min(CFG.max_len, 512)  # 不超過 DeBERTa 的最大限制
LOGGER.info(f'max_len: {CFG.max_len}')

pn_history lengths:   0%|          | 0/1000 [00:00<?, ?it/s]

pn_history max token length: 309
pn_history max token length: 309
pn_history max token length: 309
pn_history max token length: 309
pn_history max token length: 309
pn_history max token length: 309
pn_history max token length: 309
pn_history max token length: 309
pn_history max token length: 309
pn_history max token length: 309
INFO:__main__:pn_history max token length: 309


feature_text lengths:   0%|          | 0/131 [00:00<?, ?it/s]

feature_text max token length: 13
feature_text max token length: 13
feature_text max token length: 13
feature_text max token length: 13
feature_text max token length: 13
feature_text max token length: 13
feature_text max token length: 13
feature_text max token length: 13
feature_text max token length: 13
feature_text max token length: 13
INFO:__main__:feature_text max token length: 13
max_len: 325
max_len: 325
max_len: 325
max_len: 325
max_len: 325
max_len: 325
max_len: 325
max_len: 325
max_len: 325
max_len: 325
INFO:__main__:max_len: 325


# Dataset

### prepare_input
把 `pn_history`（病歷）和 `feature_text`（要找的臨床特徵）一起餵給 tokenizer：
```
[CLS] pn_history_tokens [SEP] feature_text_tokens [SEP] [PAD]...
```
模型透過 cross-attention 讓兩段文字互相參照，找出病歷中哪些部分與 feature 相關。

### create_label
建立 token-level 的訓練標籤：
```
token 屬於 special / padding / feature_text → label = -1（loss 計算時忽略）
token 屬於 pn_history 且在答案 span 內     → label = 1
token 屬於 pn_history 但不在答案 span 內   → label = 0
```

注意：label 是對 pn_history 單獨 tokenize 建立的（只取 offset_mapping），
位置 0~n 和 prepare_input 中 pn_history 的位置對齊，feature_text 位置的 label 全為 -1 故不影響 loss。

In [159]:
def prepare_input(cfg, text, feature_text):
    """把 pn_history + feature_text 一起 tokenize 成模型輸入"""
    inputs = cfg.tokenizer(
        text, feature_text,
        add_special_tokens=True,
        max_length=CFG.max_len,
        padding='max_length',
        return_offsets_mapping=False,
    )
    for k, v in inputs.items():
        inputs[k] = torch.tensor(v, dtype=torch.long)
    return inputs


def create_label(cfg, text, char_spans):
    """
    對 pn_history 單獨 tokenize，取得 offset_mapping，
    再根據 char_spans 標記每個 token 是否為答案。
    """
    encoded = cfg.tokenizer(
        text,                       # 只 tokenize pn_history
        add_special_tokens=True,
        max_length=CFG.max_len,
        padding='max_length',
        return_offsets_mapping=True, # 取得每個 token 對應的字元範圍
    )
    offset_mapping = encoded['offset_mapping']

    # sequence_ids() 回傳 None 表示 special token 或 padding，設為 -1 讓 loss 忽略
    ignore_idxes = np.where(np.array(encoded.sequence_ids()) != 0)[0]
    label = np.zeros(len(offset_mapping))
    label[ignore_idxes] = -1

    # 對每個答案 span，找出覆蓋到這個 span 的 token，標記為 1
    for start, end in char_spans:
        start_idx = -1
        end_idx   = -1
        for idx in range(len(offset_mapping)):
            # 找到第一個 token 起始位置 > span start 的前一個 token
            if (start_idx == -1) and (start < offset_mapping[idx][0]):
                start_idx = idx - 1
            # 找到第一個 token 結束位置 >= span end 的 token
            if (end_idx == -1) and (end <= offset_mapping[idx][1]):
                end_idx = idx + 1
        if start_idx == -1:
            start_idx = end_idx
        if (start_idx != -1) and (end_idx != -1):
            label[start_idx:end_idx] = 1

    return torch.tensor(label, dtype=torch.float)


class TrainDataset(Dataset):
    def __init__(self, cfg, df):
        self.cfg           = cfg
        self.feature_texts = df['feature_text'].values
        self.pn_historys   = df['pn_history'].values
        self.char_spans    = df['char_spans'].values   # list of tuples，例如 [(70,91),(176,183)]

    def __len__(self):
        return len(self.feature_texts)

    def __getitem__(self, item):
        inputs = prepare_input(self.cfg, self.pn_historys[item], self.feature_texts[item])
        label  = create_label(self.cfg, self.pn_historys[item], self.char_spans[item])
        return inputs, label

# Model

架構：`DeBERTa-v3-base` backbone + `Linear(hidden_size → 1)`

```
input_ids, attention_mask
        ↓
  DeBERTa backbone
        ↓
  last hidden states  [batch, max_len, hidden_size]
        ↓
  Dropout(0.2)
        ↓
  Linear(hidden_size → 1)
        ↓
  logits  [batch, max_len, 1]  → 每個 token 是否為答案的原始分數
```

loss 用 `BCEWithLogitsLoss`（sigmoid + BCE），只計算 label != -1 的位置。

In [160]:
class CustomModel(nn.Module):
    def __init__(self, cfg, config_path=None, pretrained=False):
        super().__init__()
        self.cfg = cfg
        # 從 pretrained 或 config 檔建立模型設定
        if config_path is None:
            self.config = AutoConfig.from_pretrained(cfg.model, output_hidden_states=True)
        else:
            self.config = torch.load(config_path)
        # pretrained=True：載入預訓練權重；pretrained=False：隨機初始化（inference 時用）
        if pretrained:
            self.model = AutoModel.from_pretrained(cfg.model, config=self.config)
        else:
            self.model = AutoModel.from_config(self.config)
        self.fc_dropout = nn.Dropout(cfg.fc_dropout)
        self.fc = nn.Linear(self.config.hidden_size, 1)  # 每個 token 輸出一個 logit
        self._init_weights(self.fc)

    def _init_weights(self, module):
        """用 DeBERTa config 的 initializer_range 初始化 Linear 層"""
        if isinstance(module, nn.Linear):
            module.weight.data.normal_(mean=0.0, std=self.config.initializer_range)
            if module.bias is not None:
                module.bias.data.zero_()
        elif isinstance(module, nn.Embedding):
            module.weight.data.normal_(mean=0.0, std=self.config.initializer_range)
            if module.padding_idx is not None:
                module.weight.data[module.padding_idx].zero_()
        elif isinstance(module, nn.LayerNorm):
            module.bias.data.zero_()
            module.weight.data.fill_(1.0)

    def feature(self, inputs):
        outputs = self.model(**inputs)
        return outputs[0]  # last hidden states: [batch, seq_len, hidden_size]

    def forward(self, inputs):
        feature = self.feature(inputs)
        return self.fc(self.fc_dropout(feature.float()))  # [batch, seq_len, 1]

# Helper Functions

- `AverageMeter`：追蹤 loss 的移動平均
- `train_fn`：一個 epoch 的訓練迴圈（含 AMP、gradient clipping）
- `valid_fn`：一個 epoch 的驗證迴圈（不更新梯度）

In [161]:
class AverageMeter:
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = self.avg = self.sum = self.count = 0

    def update(self, val, n=1):
        self.val    = val
        self.sum   += val * n
        self.count += n
        self.avg    = self.sum / self.count


def asMinutes(s):
    m = math.floor(s / 60)
    s -= m * 60
    return f'{m}m {s:.0f}s'


def timeSince(since, percent):
    now = time.time()
    s   = now - since
    rs  = s / percent - s
    return f'{asMinutes(s)} (remain {asMinutes(rs)})'


def train_fn(fold, train_loader, model, criterion, optimizer, epoch, scheduler, device):
    model.train()
    losses = AverageMeter()
    start  = time.time()
    global_step = 0

    for step, (inputs, labels) in enumerate(train_loader):
        for k, v in inputs.items():
            inputs[k] = v.to(device)
        labels     = labels.to(device)
        batch_size = labels.size(0)

        y_preds = model(inputs)   # [batch, max_len, 1]

        loss = criterion(y_preds.view(-1, 1), labels.view(-1, 1))
        loss = torch.masked_select(loss, labels.view(-1, 1) != -1).mean()

        if CFG.gradient_accumulation_steps > 1:
            loss = loss / CFG.gradient_accumulation_steps

        losses.update(loss.item(), batch_size)
        loss.backward()

        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.max_grad_norm)

        if (step + 1) % CFG.gradient_accumulation_steps == 0:
            optimizer.step()
            optimizer.zero_grad()
            global_step += 1
            if CFG.batch_scheduler:
                scheduler.step()

        if step % CFG.print_freq == 0 or step == len(train_loader) - 1:
            print(f'Epoch: [{epoch+1}][{step}/{len(train_loader)}] '
                  f'Elapsed {timeSince(start, (step+1)/len(train_loader))} '
                  f'Loss: {losses.val:.4f}({losses.avg:.4f}) '
                  f'Grad: {grad_norm:.4f}  '
                  f'LR: {scheduler.get_lr()[0]:.8f}')

    return losses.avg


def valid_fn(valid_loader, model, criterion, device):
    losses = AverageMeter()
    model.eval()
    preds = []
    start = time.time()

    for step, (inputs, labels) in enumerate(valid_loader):
        for k, v in inputs.items():
            inputs[k] = v.to(device)
        labels     = labels.to(device)
        batch_size = labels.size(0)

        with torch.no_grad():
            y_preds = model(inputs)

        loss = criterion(y_preds.view(-1, 1), labels.view(-1, 1))
        loss = torch.masked_select(loss, labels.view(-1, 1) != -1).mean()

        if CFG.gradient_accumulation_steps > 1:
            loss = loss / CFG.gradient_accumulation_steps

        losses.update(loss.item(), batch_size)
        preds.append(y_preds.sigmoid().cpu().numpy())

        if step % CFG.print_freq == 0 or step == len(valid_loader) - 1:
            print(f'EVAL: [{step}/{len(valid_loader)}] '
                  f'Elapsed {timeSince(start, (step+1)/len(valid_loader))} '
                  f'Loss: {losses.val:.4f}({losses.avg:.4f})')

    predictions = np.concatenate(preds)
    return losses.avg, predictions

# Train Loop

每個 fold 的訓練流程：
```
建立 train/valid DataLoader
→ 載入預訓練 DeBERTa（pretrained=True）
→ 設定 differential learning rate（backbone 和 head 用不同 lr）
→ 跑 5 個 epoch，每個 epoch 結束後：
   - 計算 valid loss
   - 把 token 機率轉回 char 機率 → decode 成 span → 計算 char-level F1
   - 如果 F1 有進步，存下模型權重
→ 回傳 valid fold 的預測結果（oof_df 的一部分）
```

In [ ]:
def get_scheduler(cfg, optimizer, num_train_steps):
    if cfg.scheduler == 'linear':
        return get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=cfg.num_warmup_steps,
            num_training_steps=num_train_steps,
        )
    elif cfg.scheduler == 'cosine':
        return get_cosine_schedule_with_warmup(
            optimizer,
            num_warmup_steps=cfg.num_warmup_steps,
            num_training_steps=num_train_steps,
            num_cycles=cfg.num_cycles,
        )


def get_optimizer_params(model, encoder_lr, decoder_lr, weight_decay=0.0):
    no_decay = ['bias', 'LayerNorm.bias', 'LayerNorm.weight']
    return [
        {'params': [p for n, p in model.model.named_parameters() if not any(nd in n for nd in no_decay)],
         'lr': encoder_lr, 'weight_decay': weight_decay},
        {'params': [p for n, p in model.model.named_parameters() if any(nd in n for nd in no_decay)],
         'lr': encoder_lr, 'weight_decay': 0.0},
        {'params': [p for n, p in model.named_parameters() if 'model' not in n],
         'lr': decoder_lr, 'weight_decay': 0.0},
    ]


def train_loop(folds, fold):
    LOGGER.info(f'========== fold: {fold} training ==========')

    train_folds  = folds[folds['fold'] != fold].reset_index(drop=True)
    valid_folds  = folds[folds['fold'] == fold].reset_index(drop=True)
    valid_texts  = valid_folds['pn_history'].values
    valid_labels = create_labels_for_scoring(valid_folds)

    train_dataset = TrainDataset(CFG, train_folds)
    valid_dataset = TrainDataset(CFG, valid_folds)

    train_loader = DataLoader(
        train_dataset, batch_size=CFG.batch_size, shuffle=True,
        num_workers=CFG.num_workers, pin_memory=True, drop_last=True,
    )
    valid_loader = DataLoader(
        valid_dataset, batch_size=CFG.batch_size, shuffle=False,
        num_workers=CFG.num_workers, pin_memory=True, drop_last=False,
    )

    # 診斷：確認標籤分布
    sample_inputs, sample_labels = next(iter(train_loader))
    n_pos = (sample_labels == 1).sum().item()
    n_neg = (sample_labels == 0).sum().item()
    pos_rate = n_pos / (n_pos + n_neg) if (n_pos + n_neg) > 0 else 0
    LOGGER.info(f'Label check — pos: {n_pos}, neg: {n_neg}, pos_rate: {pos_rate:.4f}')

    model = CustomModel(CFG, config_path=None, pretrained=True)
    torch.save(model.config, OUTPUT_DIR + 'config.pth')
    model.to(device)

    optimizer_parameters = get_optimizer_params(
        model, encoder_lr=CFG.encoder_lr, decoder_lr=CFG.decoder_lr,
        weight_decay=CFG.weight_decay,
    )
    optimizer = AdamW(optimizer_parameters, lr=CFG.encoder_lr, eps=CFG.eps, betas=CFG.betas)

    num_train_steps = int(len(train_folds) / CFG.batch_size * CFG.epochs)
    scheduler = get_scheduler(CFG, optimizer, num_train_steps)

    criterion  = nn.BCEWithLogitsLoss(reduction='none')
    best_score = -1.

    for epoch in range(CFG.epochs):
        start_time = time.time()

        avg_loss = train_fn(fold, train_loader, model, criterion, optimizer, epoch, scheduler, device)

        avg_val_loss, predictions = valid_fn(valid_loader, model, criterion, device)
        predictions = predictions.reshape((len(valid_folds), CFG.max_len))
        LOGGER.info(f'Max prediction prob: {predictions.max():.4f}  Mean: {predictions.mean():.4f}')

        char_probs = get_char_probs(valid_texts, predictions, CFG.tokenizer)
        results    = get_results(char_probs, th=0.5)
        preds      = get_predictions(results)
        score      = get_score(valid_labels, preds)

        elapsed = time.time() - start_time
        LOGGER.info(f'Epoch {epoch+1} - avg_train_loss: {avg_loss:.4f}  avg_val_loss: {avg_val_loss:.4f}  time: {elapsed:.0f}s')
        LOGGER.info(f'Epoch {epoch+1} - Score: {score:.4f}')

        if best_score < score:
            best_score = score
            LOGGER.info(f'Epoch {epoch+1} - Save Best Score: {best_score:.4f} Model')
            torch.save(
                {'model': model.state_dict(), 'predictions': predictions},
                OUTPUT_DIR + f"{CFG.model.replace('/', '-')}_fold{fold}_best.pth",
            )

    predictions = torch.load(
        OUTPUT_DIR + f"{CFG.model.replace('/', '-')}_fold{fold}_best.pth",
        map_location='cpu',
        weights_only=False,
    )['predictions']
    valid_folds[[i for i in range(CFG.max_len)]] = predictions

    torch.cuda.empty_cache()
    gc.collect()

    return valid_folds

# Main

跑完所有 fold 後，`oof_df` 包含全部 14,300 筆的驗證集預測結果（每筆資料只被當作一次驗證集），
用來計算整體 CV score，以及之後 inference notebook 做 threshold tuning。

In [163]:
def get_result(oof_df):
    """計算並印出這個 oof_df 的 char-level F1 score"""
    labels      = create_labels_for_scoring(oof_df)
    predictions = oof_df[[i for i in range(CFG.max_len)]].values  # token-level predictions
    char_probs  = get_char_probs(oof_df['pn_history'].values, predictions, CFG.tokenizer)
    results     = get_results(char_probs, th=0.5)
    preds       = get_predictions(results)
    score       = get_score(labels, preds)
    LOGGER.info(f'Score: {score:.4f}')


if CFG.train:
    oof_df = pd.DataFrame()

    for fold in range(CFG.n_fold):
        if fold in CFG.trn_fold:
            _oof_df = train_loop(train, fold)        # 訓練並取回這個 fold 的驗證結果
            oof_df  = pd.concat([oof_df, _oof_df])   # 累積所有 fold 的驗證結果
            LOGGER.info(f'========== fold: {fold} result ==========')
            get_result(_oof_df)                      # 印出這個 fold 的分數

    oof_df = oof_df.reset_index(drop=True)
    LOGGER.info('========== CV ==========')
    get_result(oof_df)                               # 印出全部 fold 合併後的 CV 分數

    # 存 oof_df，inference notebook 會用它來調 threshold
    oof_df.to_pickle(OUTPUT_DIR + 'oof_df.pkl')
    LOGGER.info(f'Saved oof_df to {OUTPUT_DIR}oof_df.pkl')

========== fold: 0 training ==========
========== fold: 0 training ==========
========== fold: 0 training ==========
========== fold: 0 training ==========
========== fold: 0 training ==========
========== fold: 0 training ==========
========== fold: 0 training ==========
========== fold: 0 training ==========
========== fold: 0 training ==========
========== fold: 0 training ==========
INFO:__main__:========== fold: 0 training ==========
Label check — pos: 23, neg: 3100, ignored: 2077, pos_rate: 0.0074
Label check — pos: 23, neg: 3100, ignored: 2077, pos_rate: 0.0074
Label check — pos: 23, neg: 3100, ignored: 2077, pos_rate: 0.0074
Label check — pos: 23, neg: 3100, ignored: 2077, pos_rate: 0.0074
Label check — pos: 23, neg: 3100, ignored: 2077, pos_rate: 0.0074
Label check — pos: 23, neg: 3100, ignored: 2077, pos_rate: 0.0074
Label check — pos: 23, neg: 3100, ignored: 2077, pos_rate: 0.0074
Label check — pos: 23, neg: 3100, ignored: 2077, pos_rate: 0.0074
Label check — pos: 23, neg: 3

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch: [1][0/716] Elapsed 0m 0s (remain 3m 9s) Loss: 1.1847(1.1847) Grad: 13.1233  LR: 0.00002000
Epoch: [1][100/716] Elapsed 0m 10s (remain 0m 58s) Loss: 2.1600(2.3005) Grad: 29.9623  LR: 0.00001996
Epoch: [1][200/716] Elapsed 0m 19s (remain 0m 48s) Loss: 1.7862(2.1943) Grad: 48.5544  LR: 0.00001984
Epoch: [1][300/716] Elapsed 0m 28s (remain 0m 38s) Loss: 1.3355(1.9972) Grad: 19.7607  LR: 0.00001965
Epoch: [1][400/716] Elapsed 0m 37s (remain 0m 29s) Loss: 1.6732(1.8744) Grad: 8.5598  LR: 0.00001939
Epoch: [1][500/716] Elapsed 0m 46s (remain 0m 20s) Loss: 1.5472(1.8093) Grad: 6.3719  LR: 0.00001905
Epoch: [1][600/716] Elapsed 0m 55s (remain 0m 11s) Loss: 1.7424(1.7780) Grad: 8.6380  LR: 0.00001864
Epoch: [1][700/716] Elapsed 1m 4s (remain 0m 1s) Loss: 1.5043(1.7583) Grad: 6.2225  LR: 0.00001817
Epoch: [1][715/716] Elapsed 1m 5s (remain 0m 0s) Loss: 1.3157(1.7534) Grad: 20.7595  LR: 0.00001809
EVAL: [0/178] Elapsed 0m 0s (remain 0m 29s) Loss: 1.4712(1.4712)
EVAL: [100/178] Elapsed 0m 3s

Max prediction prob: 0.6225  Mean: 0.6225
Max prediction prob: 0.6225  Mean: 0.6225
Max prediction prob: 0.6225  Mean: 0.6225
Max prediction prob: 0.6225  Mean: 0.6225
Max prediction prob: 0.6225  Mean: 0.6225
Max prediction prob: 0.6225  Mean: 0.6225
Max prediction prob: 0.6225  Mean: 0.6225
Max prediction prob: 0.6225  Mean: 0.6225
Max prediction prob: 0.6225  Mean: 0.6225
Max prediction prob: 0.6225  Mean: 0.6225
INFO:__main__:Max prediction prob: 0.6225  Mean: 0.6225


...

Epoch 4 - avg_train_loss: 1.4863  avg_val_loss: 1.4566  time: 72s
Epoch 4 - avg_train_loss: 1.4863  avg_val_loss: 1.4566  time: 72s
Epoch 4 - avg_train_loss: 1.4863  avg_val_loss: 1.4566  time: 72s
Epoch 4 - avg_train_loss: 1.4863  avg_val_loss: 1.4566  time: 72s
Epoch 4 - avg_train_loss: 1.4863  avg_val_loss: 1.4566  time: 72s
Epoch 4 - avg_train_loss: 1.4863  avg_val_loss: 1.4566  time: 72s
Epoch 4 - avg_train_loss: 1.4863  avg_val_loss: 1.4566  time: 72s
Epoch 4 - avg_train_loss: 1.4863  avg_val_loss: 1.4566  time: 72s
Epoch 4 - avg_train_loss: 1.4863  avg_val_loss: 1.4566  time: 72s
Epoch 4 - avg_train_loss: 1.4863  avg_val_loss: 1.4566  time: 72s
INFO:__main__:Epoch 4 - avg_train_loss: 1.4863  avg_val_loss: 1.4566  time: 72s
Epoch 4 - Score: 0.0337
Epoch 4 - Score: 0.0337
Epoch 4 - Score: 0.0337
Epoch 4 - Score: 0.0337
Epoch 4 - Score: 0.0337
Epoch 4 - Score: 0.0337
Epoch 4 - Score: 0.0337
Epoch 4 - Score: 0.0337
Epoch 4 - Score: 0.0337
Epoch 4 - Score: 0.0337
INFO:__main__:Epoch 

Epoch: [5][0/715] Elapsed 0m 0s (remain 2m 55s) Loss: 1.5535(1.5535) Grad: 10.3891  LR: 0.00000191
Epoch: [5][100/715] Elapsed 0m 9s (remain 0m 55s) Loss: 1.6236(1.4958) Grad: 10.5224  LR: 0.00000143
Epoch: [5][200/715] Elapsed 0m 18s (remain 0m 46s) Loss: 1.5276(1.4980) Grad: 5.4430  LR: 0.00000101
Epoch: [5][300/715] Elapsed 0m 27s (remain 0m 37s) Loss: 1.4186(1.4974) Grad: 6.5524  LR: 0.00000066
Epoch: [5][400/715] Elapsed 0m 36s (remain 0m 28s) Loss: 1.1135(1.4878) Grad: 27.8659  LR: 0.00000038
Epoch: [5][500/715] Elapsed 0m 44s (remain 0m 19s) Loss: 1.3110(1.4846) Grad: 13.9051  LR: 0.00000018
Epoch: [5][600/715] Elapsed 0m 53s (remain 0m 10s) Loss: 1.2994(1.4838) Grad: 17.2403  LR: 0.00000005
Epoch: [5][700/715] Elapsed 1m 2s (remain 0m 1s) Loss: 1.3756(1.4806) Grad: 8.4486  LR: 0.00000000
Epoch: [5][714/715] Elapsed 1m 3s (remain 0m 0s) Loss: 1.1139(1.4805) Grad: 27.8229  LR: 0.00000000
EVAL: [0/179] Elapsed 0m 0s (remain 0m 30s) Loss: 1.4894(1.4894)
EVAL: [100/179] Elapsed 0m 3

Max prediction prob: 0.6433  Mean: 0.6433
Max prediction prob: 0.6433  Mean: 0.6433
Max prediction prob: 0.6433  Mean: 0.6433
Max prediction prob: 0.6433  Mean: 0.6433
Max prediction prob: 0.6433  Mean: 0.6433
Max prediction prob: 0.6433  Mean: 0.6433
Max prediction prob: 0.6433  Mean: 0.6433
Max prediction prob: 0.6433  Mean: 0.6433
Max prediction prob: 0.6433  Mean: 0.6433
Max prediction prob: 0.6433  Mean: 0.6433
INFO:__main__:Max prediction prob: 0.6433  Mean: 0.6433


EVAL: [178/179] Elapsed 0m 5s (remain 0m 0s) Loss: 1.2871(1.4578)


Epoch 5 - avg_train_loss: 1.4805  avg_val_loss: 1.4578  time: 72s
Epoch 5 - avg_train_loss: 1.4805  avg_val_loss: 1.4578  time: 72s
Epoch 5 - avg_train_loss: 1.4805  avg_val_loss: 1.4578  time: 72s
Epoch 5 - avg_train_loss: 1.4805  avg_val_loss: 1.4578  time: 72s
Epoch 5 - avg_train_loss: 1.4805  avg_val_loss: 1.4578  time: 72s
Epoch 5 - avg_train_loss: 1.4805  avg_val_loss: 1.4578  time: 72s
Epoch 5 - avg_train_loss: 1.4805  avg_val_loss: 1.4578  time: 72s
Epoch 5 - avg_train_loss: 1.4805  avg_val_loss: 1.4578  time: 72s
Epoch 5 - avg_train_loss: 1.4805  avg_val_loss: 1.4578  time: 72s
Epoch 5 - avg_train_loss: 1.4805  avg_val_loss: 1.4578  time: 72s
INFO:__main__:Epoch 5 - avg_train_loss: 1.4805  avg_val_loss: 1.4578  time: 72s
Epoch 5 - Score: 0.0337
Epoch 5 - Score: 0.0337
Epoch 5 - Score: 0.0337
Epoch 5 - Score: 0.0337
Epoch 5 - Score: 0.0337
Epoch 5 - Score: 0.0337
Epoch 5 - Score: 0.0337
Epoch 5 - Score: 0.0337
Epoch 5 - Score: 0.0337
Epoch 5 - Score: 0.0337
INFO:__main__:Epoch 

In [164]:
# 用完美預測測試 scoring pipeline
valid_folds = train[train['fold'] == 0].reset_index(drop=True)
valid_texts  = valid_folds['pn_history'].values
valid_labels = create_labels_for_scoring(valid_folds)

print(f"非空 ground truth 筆數: {sum(len(t) > 0 for t in valid_labels)}")

  # 把正確答案位置的機率設成 1.0（模擬完美預測）
perfect_char_probs = [np.zeros(len(t)) for t in valid_texts]
for i, truth in enumerate(valid_labels):
    for start, end in truth:
        perfect_char_probs[i][start:end] = 1.0

results = get_results(perfect_char_probs, th=0.5)
preds   = get_predictions(results)
score   = get_score(valid_labels, preds)
print(f"完美預測的 Score（應該接近 1.0）: {score:.4f}")

非空 ground truth 筆數: 1976
完美預測的 Score（應該接近 1.0）: 0.9634
